# Adquisición y Normalización de Datos — TechStore

## Objetivo

Simular un escenario real donde los datos de ventas provienen de diferentes fuentes y formatos.

- Las transacciones de ventas se encuentran en un archivo CSV.
- El catálogo de productos se encuentra en un archivo Excel.

El objetivo es integrar ambas fuentes utilizando Pandas, realizar una limpieza básica de los datos, calcular el total de cada venta y exportar el dataset final en formato Parquet.

### Etapas del proceso

1. Adquisición de los datos
2. Transformación y limpieza
3. Integración mediante `merge`
4. Normalización mediante el cálculo de `total_venta`
5. Validación
6. Exportación a Parquet

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
RUTA_TRANSACCIONES = Path("transacciones_techstore.csv")
RUTA_PRODUCTOS = Path("productos_techstore.xlsx")
RUTA_SALIDA = Path("ventas_techstore_consolidado.parquet")

print("CSV disponible:", RUTA_TRANSACCIONES.exists())
print("Excel disponible:", RUTA_PRODUCTOS.exists())

CSV disponible: True
Excel disponible: True


###1) Adquisición de datos

In [3]:
transacciones = pd.read_csv(RUTA_TRANSACCIONES)

print("Transacciones:", transacciones.shape)
transacciones.head()

Transacciones: (21, 4)


,id_transaccion,id_producto,cantidad,fecha
0,1001,101,2.0,2026-01-05
1,1002,103,1.0,05/01/2026
2,1003,102,3.0,2026-01-07
3,1004,105,1.0,08/01/2026
4,1005,101,1.0,2026-01-10


In [4]:
productos = pd.read_excel(RUTA_PRODUCTOS)

print("Productos:", productos.shape)
productos.head()

Productos: (8, 3)


,id_producto,nombre_producto,precio_unitario
0,101,Teclado inalámbrico,25000
1,102,Mouse óptico,12000
2,103,Auriculares Bluetooth,35000
3,104,Webcam HD,28000
4,105,Monitor 24 pulgadas,180000


###2) Exploración inicial

In [5]:
transacciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_transaccion  21 non-null     int64  
 1   id_producto     21 non-null     int64  
 2   cantidad        20 non-null     float64
 3   fecha           21 non-null     object 
dtypes: float64(1), int64(2), object(1)
memory usage: 804.0+ bytes


In [6]:
print("Nulos por columna:")
print(productos.isnull().sum())

Nulos por columna:
id_producto        0
nombre_producto    0
precio_unitario    0
dtype: int64


In [7]:
print("Tipos de datos:")
print(productos.dtypes)

Tipos de datos:
id_producto         int64
nombre_producto    object
precio_unitario     int64
dtype: object


### Observaciones iniciales

La exploración de los datos permitió identificar algunos aspectos que deberán ser tratados antes de integrar ambas fuentes:

- La columna `fecha` de las transacciones está almacenada como texto (`object`) y deberá convertirse a `datetime`.
- La columna `cantidad` contiene un valor nulo. Como no es posible determinar la cantidad vendida, se eliminará esa transacción en lugar de imputar un valor.
- El catálogo de productos no presenta valores nulos relevantes.
- El tipo de dato de `id_producto` deberá verificarse en ambas fuentes antes de realizar el `merge`.

###3.1) Tranformaciones
- Transformar fecha

In [8]:
transacciones["fecha"].head(10).tolist()

['2026-01-05',
 '05/01/2026',
 '2026-01-07',
 '08/01/2026',
 '2026-01-10',
 '11/01/2026',
 '2026-01-12',
 '13/01/2026',
 '2026-01-14',
 '15/01/2026']

In [9]:
transacciones["fecha"] = pd.to_datetime(
    transacciones["fecha"],
    format="mixed",
    dayfirst=True
)

In [10]:
print(transacciones["fecha"].dtype)

datetime64[ns]


In [11]:
transacciones[["id_transaccion", "fecha"]].head(10)

,id_transaccion,fecha
0,1001,2026-01-05
1,1002,2026-01-05
2,1003,2026-01-07
3,1004,2026-01-08
4,1005,2026-01-10
5,1006,2026-01-11
6,1007,2026-01-12
7,1008,2026-01-13
8,1009,2026-01-14
9,1010,2026-01-15


###3.2) Tranformaciones
- Manejo de valores nulos

In [12]:
nulos_antes = transacciones["cantidad"].isnull().sum()

print(f"Valores nulos en 'cantidad': {nulos_antes}")

Valores nulos en 'cantidad': 1


In [13]:
transacciones = transacciones.dropna(subset=["cantidad"])

In [15]:
transacciones["cantidad"] = transacciones["cantidad"].astype(int)

In [16]:
print(transacciones["cantidad"].dtype)

int64


In [17]:
print(f"Filas restantes: {len(transacciones)}")

Filas restantes: 20


###3.3) Tranformaciones
- Duplicados

In [18]:
duplicados = transacciones.duplicated().sum()

print(f"Filas duplicadas encontradas: {duplicados}")

Filas duplicadas encontradas: 1


In [19]:
transacciones = transacciones.drop_duplicates()

### Limpieza de datos

Durante la transformación se realizaron las siguientes acciones:

- La columna `fecha` fue convertida de `object` a `datetime`.
- Se eliminó una transacción con `cantidad` nula, ya que no era posible determinar un valor correcto para imputar.
- Se detectó y eliminó una fila duplicada.
- La columna `cantidad` fue convertida a tipo entero después de eliminar el valor nulo.

###4) Merge

In [20]:
print("id_producto en transacciones:", transacciones["id_producto"].dtype)
print("id_producto en productos:", productos["id_producto"].dtype)

id_producto en transacciones: int64
id_producto en productos: int64


In [21]:
ventas = transacciones.merge(
    productos,
    on="id_producto",
    how="left",
    indicator=True
)

In [22]:
print(ventas["_merge"].value_counts())

_merge
both          18
left_only      1
right_only     0
Name: count, dtype: int64


In [23]:
huerfanas = ventas[ventas["_merge"] == "left_only"]

huerfanas[[
    "id_transaccion",
    "id_producto",
    "cantidad",
    "fecha"
]]

,id_transaccion,id_producto,cantidad,fecha
9,1011,999,2,2026-01-16


In [24]:
ventas = (
    ventas[ventas["_merge"] == "both"]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

In [25]:
print(f"Filas finales tras el merge: {len(ventas)}")

Filas finales tras el merge: 18


###5) Crear `Total_venta`

In [26]:
ventas["total_venta"] = (
    ventas["cantidad"] * ventas["precio_unitario"]
)

In [27]:
ventas[
    [
        "id_transaccion",
        "nombre_producto",
        "cantidad",
        "precio_unitario",
        "total_venta"
    ]
].head(10)

,id_transaccion,nombre_producto,cantidad,precio_unitario,total_venta
0,1001,Teclado inalámbrico,2,25000.0,50000.0
1,1002,Auriculares Bluetooth,1,35000.0,35000.0
2,1003,Mouse óptico,3,12000.0,36000.0
3,1004,Monitor 24 pulgadas,1,180000.0,180000.0
4,1005,Teclado inalámbrico,1,25000.0,25000.0
5,1007,Hub USB-C,2,22000.0,44000.0
6,1008,Mouse óptico,1,12000.0,12000.0
7,1009,Auriculares Bluetooth,2,35000.0,70000.0
8,1010,Parlante portátil,1,45000.0,45000.0
9,1012,Monitor 24 pulgadas,1,180000.0,180000.0


In [28]:
assert ventas.isnull().sum().sum() == 0, \
    "Hay valores nulos en el dataset final"

In [29]:
assert ventas.duplicated().sum() == 0, \
    "Hay filas duplicadas en el dataset final"

In [30]:
assert (ventas["total_venta"] >= 0).all(), \
    "Hay valores negativos en total_venta"

In [31]:
assert pd.api.types.is_datetime64_any_dtype(ventas["fecha"]), \
    "La columna fecha no es datetime"

In [32]:
print("Todas las validaciones fueron exitosas ✅")
print(f"Cantidad de transacciones finales: {len(ventas)}")
print(f"Total de ventas: ${ventas['total_venta'].sum():,.0f}")

Todas las validaciones fueron exitosas ✅
Cantidad de transacciones finales: 18
Total de ventas: $1,120,000


###6) Exportación a Parquet

In [33]:
!pip install pyarrow -q

In [34]:
RUTA_SALIDA = Path("ventas_techstore_consolidado.parquet")

ventas.to_parquet(
    RUTA_SALIDA,
    index=False
)

print(f"Archivo exportado correctamente: {RUTA_SALIDA}")

Archivo exportado correctamente: ventas_techstore_consolidado.parquet


In [35]:
verificacion = pd.read_parquet(RUTA_SALIDA)

print("Parquet leído correctamente ✅")
print(verificacion.shape)

Parquet leído correctamente ✅
(18, 7)


In [36]:
print(verificacion.columns.tolist())

['id_transaccion', 'id_producto', 'cantidad', 'fecha', 'nombre_producto', 'precio_unitario', 'total_venta']


In [37]:
verificacion.head()

,id_transaccion,id_producto,cantidad,fecha,nombre_producto,precio_unitario,total_venta
0,1001,101,2,2026-01-05,Teclado inalámbrico,25000.0,50000.0
1,1002,103,1,2026-01-05,Auriculares Bluetooth,35000.0,35000.0
2,1003,102,3,2026-01-07,Mouse óptico,12000.0,36000.0
3,1004,105,1,2026-01-08,Monitor 24 pulgadas,180000.0,180000.0
4,1005,101,1,2026-01-10,Teclado inalámbrico,25000.0,25000.0


## Conclusiones

Se integraron correctamente los datos de transacciones y el catálogo de productos provenientes de archivos CSV y Excel.

Durante el proceso se realizaron las siguientes tareas:

- Conversión de la columna `fecha` a formato `datetime`.
- Eliminación de registros con valores nulos en `cantidad`.
- Eliminación de registros duplicados.
- Verificación de compatibilidad del tipo de dato de `id_producto`.
- Integración de ambas fuentes mediante un `merge`.
- Identificación y exclusión de una transacción cuyo producto no se encontraba en el catálogo.
- Cálculo de la columna `total_venta`.
- Validación de la calidad del dataset final.
- Exportación del resultado consolidado en formato Parquet.